#### Visão Geral
##### Schema : silver
##### Table : case_logistica_entregas

| Detalhe | Informação |
|---------|------------|
| Criado Originalmente Por | Wellikiandre Bosich |
| Tabela de Dados de Saída | `{environment}.silver.case_logistica_entregas` |
| Origem Fonte de Dados de Entrada | Camada bronze |
| Destino Fonte de Dados de Saída | Camada silver |

#### Histórico

| Data       | Desenvolvido Por         | Motivo                                         |
|:----------:|--------------------------|-----------------------------------------------|
| 04/06/2026 | Wellikiandre Bosich    | Criação do notebook e estruturação de KPI logísticos de transporte na Silver. |

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
sistema = 'case'
table_name = 'logistica_entregas'
input_path = f"{var_bronze}/{sistema}/{table_name}/data"
output_path_data = f"{var_silver}/{sistema}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_silver_schema}.{sistema}_{table_name}'

In [ ]:
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.types import DecimalType

df_bronze = spark.read.format("delta").load(input_path)

df_clean = (
    df_bronze
    .withColumn("timestamps_delivered_at", to_timestamp(col("timestamps_delivered_at")))
    .withColumn("timestamps_shipped_at", to_timestamp(col("timestamps_shipped_at")))
    .withColumn("cost", col("cost").cast(DecimalType(10, 2)))
    .select(
        col("delivery_id").cast("string").alias("id_entrega"),
        col("order_ref").cast("integer").alias("id_pedido"),
        col("delivery_status").cast("string").alias("status_entrega"),
        col("carrier_name").cast("string").alias("transportadora"),
        col("carrier_mode").cast("string").alias("modalidade_transporte"),
        col("cost").alias("custo_frete"),
        col("destination_city").cast("string").alias("cidade_destino"),
        col("destination_state").cast("string").alias("uf_destino"),
        col("timestamps_shipped_at").alias("data_envio"),
        col("timestamps_delivered_at").alias("data_entrega")
    )
    .filter(col("id_entrega").isNotNull() & col("id_pedido").isNotNull())
    .dropDuplicates(["id_entrega"])
)

In [ ]:
process_data(
    df_write=df_clean,
    tipo_carga='delta',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['status_entrega'],
    chave_upsert='id_entrega'
)